# Singapore price-rent stochastic model

Offline analysis notebook. Reads the transaction data and writes four small precomputed
JSON artifacts into `data/` that the static web app loads and renders. **All statistics happen
here** - the browser does zero model fitting.

Artifacts written:

- `series.json` - quarterly sale-PSF and rental-PSF medians per district x size band.
- `model_params.json` - fitted price-rent econometrics per segment (correlation, cointegration,
  VECM error-correction / lead-lag, OU-on-yield calibration, structural-break scan) with an
  uncertainty/adequacy flag on every block.
- `mc_grid.json` - precomputed Monte-Carlo terminal-wealth summaries (buy vs ETF) plus break-even
  appreciation, on a discretised grid the client interpolates.
- `model_meta.json` - provenance (data range, model specs, MC seed/paths, grid axes).

**Data limitation (read this).** With ~5 years (~20 quarters, one market regime) the cointegration /
VECM / OU estimates are statistically weak. They are computed and stored for forward-readiness, and
every block is flagged `low_confidence` until the history is long enough. Drop in a longer REALIS
series (set the config cell) and re-run top-to-bottom; the flags clear automatically.

**To regenerate:** Kernel -> Restart & Run All. Libraries: pandas, numpy, statsmodels, scipy only.

## 0. Imports

In [1]:
#!/usr/bin/env python3
"""
Core analysis logic for the Singapore price-rent stochastic model.
Developed as a plain script for fast iteration/verification, then ported
verbatim into the notebook (sg_price_rent_model.ipynb). Keep this and the
notebook in sync during development; the notebook is the shipped artifact.

Libraries: pandas, numpy, statsmodels, scipy only.
"""
import json, os, math
import numpy as np
import pandas as pd
from statsmodels.tsa.stattools import coint, adfuller
from statsmodels.tsa.vector_ar.vecm import VECM, coint_johansen
import statsmodels.api as sm

## 1. Configuration

The only cell you edit on a data refresh. Point `HISTORICAL_*_SOURCE` at the long history, set `DATE_RANGE`, leave `FIT_LEVEL='auto'`.

In [2]:
DATA_DIR = "data"   # relative to this notebook (run with cwd = singapore-apartment/)
OUT_DIR  = "data"

HISTORICAL_SALES_SOURCE = os.path.join(DATA_DIR, "sales.json")  # today: the live app file; later: point at the long REALIS export
HISTORICAL_RENT_SOURCE  = os.path.join(DATA_DIR, "rent.json")
SOURCE_KIND   = "app_json"        # "app_json" (current array-of-arrays) | "realis_csv" (future)
DATE_RANGE    = None              # None = all; or ("2006-Q1","2026-Q2")
FIT_LEVEL     = "auto"            # "auto" | "district" | "district_band"
# Under "auto", the fit only SUBDIVIDES a district into per-band series when every band cell
# has at least this many REAL (non-interpolated) quarters. Set to the same bar as a robust fit:
# subdividing thin data just makes each series noisier. Today (~20q) this keeps us at district
# level; a 20-year refresh (~80q/band) auto-upgrades to district_band.
MIN_QUARTERS_FIT     = 40
MAX_INTERP_FRAC_BAND = 0.15       # and no more than this fraction of a band series interpolated
MIN_RECOMMENDED_QUARTERS = 40     # below this -> data_adequacy flag = low_confidence

MC_N_PATHS = 5000
MC_SEED    = 42
DT_YEARS   = 0.25                 # quarterly step

# MC grid axes
AX_HORIZON = [5, 10, 15, 20, 25, 30]
AX_LTV     = [0, 25, 50, 55, 60, 65, 70, 75, 80]
AX_YIELD_BUCKET = ["low", "mid", "high"]
AX_DRIFT   = [-2, 0, 2, 4, 6]     # appreciation CAGR %, user-overlaid
AX_ETF     = [3, 5, 7, 9, 11]     # opportunity-cost/ETF CAGR %, user-overlaid

# Deterministic finance assumptions for the terminal-wealth model (documented in meta).
# Kept consistent with build_data.py / app.js where they overlap.
MORTGAGE_RATE_PCT = 3.0           # forward mortgage rate assumption for the MC (grid is not per-rate)
LOAN_TENURE_YRS   = 30
MCST_MO   = 500
INS_YR    = 300
AGENT_PCT = 0.042                 # leasing agent, % of effective annual rent
REPAIR_PCT= 0.02
VAC_MO    = 1
BASE_INCOME = 0                   # buyer's other chargeable income (marginal-rate base)
ABSD_PCT  = 0.0                   # reference buyer = SC 1st property; dollar outputs are provisional
LEGAL_MISC= 3000
SELL_AGENT_PCT = 0.02             # exit: selling agent commission
INFLATION_PCT  = 2.0             # for real (inflation-adjusted) terminal wealth AND opex growth
EXPENSE_GROWTH_PCT = INFLATION_PCT  # rent/expense growth (rent grows via yield*price; fixed opex via this)

BANDS = ["A", "B", "C"]
BAND_EDGES = {"A": (350, 550), "B": (550, 750), "C": (750, 1000)}

## 2. Loader / adapter

Reads the current app JSON shape. `load_realis_csv` is a stub for the future separate long-history export - implement its column mapping when that file exists.

In [3]:
def _band(sqft):
    if sqft < 550: return "A"
    if sqft < 750: return "B"
    return "C"

def load_app_json(sales_path, rent_path):
    """Current shape:
       sales.json: [district, project, tenure, sqft, psf, price, "YYYY-MM"]
       rent.json:  [district, project, bed, sqft, rent, "YYYY-MM"]
    """
    with open(sales_path) as f: sraw = json.load(f)
    with open(rent_path)  as f: rraw = json.load(f)
    sales = pd.DataFrame(sraw, columns=["district","project","tenure","sqft","psf","price","ym"])
    rent  = pd.DataFrame(rraw, columns=["district","project","bed","sqft","rent","ym"])
    for df in (sales, rent):
        # defensive shape validation - fail loudly, never silently coerce garbage
        if df["ym"].str.match(r"^\d{4}-\d{2}$").mean() < 0.99:
            raise ValueError("Unexpected date format in source; expected 'YYYY-MM'")
    sales["rent_psf"] = np.nan
    rent["rent_psf"]  = rent["rent"] / rent["sqft"]     # monthly rent per sqft
    return sales, rent

def load_realis_csv(sales_path, rent_path):
    """STUB for the future separate long-history REALIS export. Not exercised today.
    When a real export arrives, map its columns to the same frame shape returned by
    load_app_json (district, project, sqft, psf, price, ym[, rent, rent_psf]) here."""
    raise NotImplementedError("REALIS CSV adapter: implement column mapping when the long export is available.")

def load_data():
    if SOURCE_KIND == "app_json":
        return load_app_json(HISTORICAL_SALES_SOURCE, HISTORICAL_RENT_SOURCE)
    elif SOURCE_KIND == "realis_csv":
        return load_realis_csv(HISTORICAL_SALES_SOURCE, HISTORICAL_RENT_SOURCE)
    raise ValueError(f"Unknown SOURCE_KIND {SOURCE_KIND!r}")

def to_quarter(ym):
    y, m = ym.split("-"); y = int(y); m = int(m)
    return f"{y}-Q{(m-1)//3 + 1}"

def quarter_key(q):
    y, qq = q.split("-Q"); return int(y)*4 + int(qq)

def quarter_range(qmin, qmax):
    """Inclusive list of all quarters between qmin and qmax (no gaps in the axis)."""
    a, b = quarter_key(qmin), quarter_key(qmax)
    out = []
    for k in range(a, b+1):
        y, qq = divmod(k-1, 4); out.append(f"{y}-Q{qq+1}")
    return out

## 3. Quarterly aggregation (builds `series.json`)

In [4]:
def add_quarter_band(sales, rent):
    for df in (sales, rent):
        df["q"] = df["ym"].map(to_quarter)
        df["band"] = df["sqft"].map(_band)
    if DATE_RANGE is not None:
        lo, hi = quarter_key(DATE_RANGE[0]), quarter_key(DATE_RANGE[1])
        sales = sales[sales["q"].map(quarter_key).between(lo, hi)].copy()
        rent  = rent[rent["q"].map(quarter_key).between(lo, hi)].copy()
    return sales, rent

def build_series(sales, rent):
    """Quarterly sale-PSF and rental-PSF medians per district x band, aligned to a
    single shared quarter axis (parallel arrays; null where a quarter has no txns)."""
    qs = sorted(set(sales["q"]) | set(rent["q"]), key=quarter_key)
    quarters_all = quarter_range(qs[0], qs[-1])   # contiguous axis, holes become null
    qidx = {q: i for i, q in enumerate(quarters_all)}
    nQ = len(quarters_all)

    districts = sorted(set(sales["district"]) | set(rent["district"]))
    sale_g = sales.groupby(["district","band","q"])["psf"].agg(["median","size"])
    rent_g = rent.groupby(["district","band","q"])["rent_psf"].agg(["median","size"])

    series = []
    for d in districts:
        for b in BANDS:
            sp = [None]*nQ; sn = [0]*nQ; rp = [None]*nQ; rn = [0]*nQ; gy = [None]*nQ
            any_data = False
            for q in quarters_all:
                i = qidx[q]
                if (d,b,q) in sale_g.index:
                    sp[i] = round(float(sale_g.loc[(d,b,q),"median"]), 1)
                    sn[i] = int(sale_g.loc[(d,b,q),"size"]); any_data = True
                if (d,b,q) in rent_g.index:
                    rp[i] = round(float(rent_g.loc[(d,b,q),"median"]), 3)
                    rn[i] = int(rent_g.loc[(d,b,q),"size"]); any_data = True
                if sp[i] is not None and rp[i] is not None and sp[i] > 0:
                    gy[i] = round(rp[i]*12.0/sp[i], 4)   # annualized gross yield at PSF level
            if any_data:
                series.append(dict(district=d, band=b, sale_psf_median=sp, sale_n=sn,
                                   rent_psf_median=rp, rent_n=rn, gross_yield=gy))
    return dict(quarters_all=quarters_all, bands=BAND_EDGES, series=series), quarters_all, districts

## 4. Segment quarterly series for the fits

In [5]:
def district_quarterly(sales, rent, district, quarters_all, band=None):
    """Return a DataFrame indexed by the shared quarter axis with columns
    sale_psf, rent_psf, yield, logp, logr. Interior single-quarter gaps in the
    log series are linearly interpolated (flagged); leading/trailing NaN dropped.
    band=None pools all bands (district-level fit); band='A'/'B'/'C' restricts to
    that size band (used only when FIT_LEVEL resolves to district_band)."""
    ss = sales[sales["district"]==district]; rr = rent[rent["district"]==district]
    if band is not None:
        ss = ss[ss["band"]==band]; rr = rr[rr["band"]==band]
    s = ss.groupby("q")["psf"].median()
    r = rr.groupby("q")["rent_psf"].median()
    df = pd.DataFrame(index=quarters_all)
    df["sale_psf"] = [s.get(q, np.nan) for q in quarters_all]
    df["rent_psf"] = [r.get(q, np.nan) for q in quarters_all]
    # trim to the span where BOTH exist at the ends
    both = df.dropna()
    if len(both) < 2:
        return None, 0
    first, last = both.index[0], both.index[-1]
    lo = quarters_all.index(first); hi = quarters_all.index(last)
    df = df.iloc[lo:hi+1].copy()
    n_interp = int(df["sale_psf"].isna().sum() + df["rent_psf"].isna().sum())
    df["logp"] = np.log(df["sale_psf"]).interpolate(method="linear")
    df["logr"] = np.log(df["rent_psf"]).interpolate(method="linear")
    df["yield"] = np.exp(df["logr"])*12.0/np.exp(df["logp"])
    return df, n_interp

## 5. Econometrics

Correlation, Engle-Granger + Johansen cointegration, VECM (error-correction, lead-lag), OU-on-yield calibration, and a Quandt-Andrews structural-break scan. Every estimate carries a standard error / CI, and short windows are flagged.

In [6]:
def fit_correlation(logp, logr):
    dp = np.diff(logp); dr = np.diff(logr)
    n = len(dp)
    if n < 3 or np.std(dp)==0 or np.std(dr)==0:
        return dict(price_rent_return_corr=None, ci95=None, n=n)
    rho = float(np.corrcoef(dp, dr)[0,1])
    # Fisher z 95% CI
    z = np.arctanh(np.clip(rho, -0.999, 0.999)); se = 1.0/math.sqrt(n-3)
    lo, hi = math.tanh(z-1.96*se), math.tanh(z+1.96*se)
    return dict(price_rent_return_corr=round(rho,4), ci95=[round(lo,4),round(hi,4)], n=n)

def fit_cointegration(logp, logr):
    out = {}
    n = len(logp)
    # DETERMINISTIC, capped lag - NOT autolag='aic'. On short series AIC lag-hunting picks
    # absurd lags (e.g. 9 on 20 obs -> degenerate stat=0) and, being data-driven, makes the
    # fitted p-value jump between data refreshes, defeating the provenance/diffability goal.
    # Fixed rule: 1 lag for short history, 2 once the series is long. Recorded in output.
    eg_lag = 1 if n < 40 else 2
    try:
        eg_t, eg_p, eg_crit = coint(logp, logr, trend="c", maxlag=eg_lag, autolag=None)
        out["engle_granger"] = dict(stat=round(float(eg_t),4), pvalue=round(float(eg_p),4), lag=eg_lag,
            critical_values={"1%":round(float(eg_crit[0]),3),"5%":round(float(eg_crit[1]),3),"10%":round(float(eg_crit[2]),3)})
        eg_coint = bool(eg_p < 0.05)
    except Exception as e:
        out["engle_granger"] = dict(error=str(e)); eg_coint = False
    joh_coint = None
    try:
        mat = np.column_stack([logp, logr])
        joh = coint_johansen(mat, det_order=0, k_ar_diff=1)
        trace = joh.lr1.tolist(); crit5 = joh.cvt[:,1].tolist()  # 5% column
        rank = int(sum(1 for i in range(len(trace)) if trace[i] > crit5[i]))
        out["johansen"] = dict(trace_stat=[round(x,3) for x in trace],
            trace_crit_5pct=[round(x,3) for x in crit5], rank=rank)
        joh_coint = bool(rank >= 1)
    except Exception as e:
        out["johansen"] = dict(error=str(e))
    out["cointegrated_at_5pct"] = bool(eg_coint or (joh_coint is True))
    return out

def fit_vecm(df, coint_established):
    """Fit a VECM (rank 1) to [logp, logr]; extract adjustment (alpha) with SEs,
    cointegrating vector (beta, normalized on price), current error-correction gap,
    and a rent->price Granger lead-lag. Always computed for forward-readiness;
    'fitted' reflects whether cointegration was established at 5%."""
    mat = df[["logp","logr"]].values
    res = dict(fitted=bool(coint_established), lag_order=1)
    try:
        model = VECM(mat, k_ar_diff=1, coint_rank=1, deterministic="ci")
        vr = model.fit()
        alpha = vr.alpha.flatten()          # (2,1) -> [alpha_price, alpha_rent]
        beta  = vr.beta.flatten()           # normalized so beta[0]=1
        alpha_se = vr.stderr_alpha.flatten() if hasattr(vr,"stderr_alpha") else [None,None]
        beta_norm = beta / beta[0] if beta[0]!=0 else beta
        # current EC gap = beta' * [logp_last, logr_last] (+ const term absorbed): use deviation of
        # the cointegrating combination from its sample mean = live "stretched vs rent-justified" gauge
        ci_series = mat @ beta_norm
        current_gap = float(ci_series[-1] - ci_series.mean())
        res["error_correction"] = dict(
            alpha_price=round(float(alpha[0]),4), alpha_price_se=(round(float(alpha_se[0]),4) if alpha_se[0] is not None else None),
            alpha_rent=round(float(alpha[1]),4),  alpha_rent_se=(round(float(alpha_se[1]),4) if alpha_se[1] is not None else None),
            beta=[round(float(beta_norm[0]),4), round(float(beta_norm[1]),4)],
            current_gap=round(current_gap,4))
    except Exception as e:
        res["error_correction"] = dict(error=str(e))
    # lead-lag: does rent Granger-cause price? (rent leads price if significant)
    try:
        from statsmodels.tsa.stattools import grangercausalitytests
        gc = grangercausalitytests(df[["logp","logr"]].values, maxlag=1, verbose=False)
        pval = float(gc[1][0]["ssr_ftest"][1])
        res["lead_lag"] = dict(rent_leads_price=bool(pval<0.05), granger_pvalue=round(pval,4))
    except Exception as e:
        res["lead_lag"] = dict(rent_leads_price=None, granger_pvalue=None, error=str(e))
    return res

def fit_ou_yield(y):
    """Calibrate OU dy = lambda(mu - y)dt + sigma dW from the quarterly yield series
    via exact-discretization AR(1): y_{t+1} = b y_t + c + eps, dt=0.25yr.
      b = e^{-lambda dt};  mu = c/(1-b);  sigma^2 = Var(eps)*2 lambda/(1-b^2).
    SEs on b,c from OLS; lambda/mu SEs via delta method."""
    y = np.asarray(y, dtype=float)
    y0, y1 = y[:-1], y[1:]
    n = len(y0)
    X = sm.add_constant(y0)
    ols = sm.OLS(y1, X).fit()
    c, b = ols.params[0], ols.params[1]
    c_se, b_se = ols.bse[0], ols.bse[1]
    resid_var = np.var(ols.resid, ddof=2)
    dt = DT_YEARS
    fitted_ok = (0 < b < 1)
    if fitted_ok:
        lam = -math.log(b)/dt
        mu  = c/(1-b)
        sigma = math.sqrt(max(resid_var,1e-12) * 2*lam/(1-b*b))
        half_life = math.log(2)/lam
        lam_se = (b_se/(b*dt)) if b>0 else None          # d lam/db = -1/(b dt)
        # mu = c/(1-b); delta method with cov(c,b)
        cov = ols.cov_params()
        dmu_dc = 1/(1-b); dmu_db = c/((1-b)**2)
        var_mu = dmu_dc**2*cov[0,0] + dmu_db**2*cov[1,1] + 2*dmu_dc*dmu_db*cov[0,1]
        mu_se = math.sqrt(max(var_mu,0))
    else:
        # non-reverting fit (b>=1 or <=0): flag; fall back to sample mean, weak reversion
        lam = 0.05/dt if b>=1 else 0.5/dt
        mu = float(np.mean(y)); sigma = math.sqrt(max(resid_var,1e-12))/dt**0.5
        half_life = math.log(2)/lam; lam_se=None; mu_se=float(np.std(y)/math.sqrt(n))
    return dict(long_run_yield=round(mu,5), long_run_yield_se=(round(mu_se,5) if mu_se else None),
        lambda_=round(lam,4), lambda_se=(round(lam_se,4) if lam_se else None),
        sigma=round(sigma,5), half_life_years=round(half_life,3),
        ar1_b=round(float(b),4), ar1_b_se=round(float(b_se),4), mean_reverting=bool(fitted_ok),
        resid_sd=round(math.sqrt(max(resid_var,1e-12)),6))

def fit_structural_breaks(y):
    """Quandt-Andrews supF single-break scan on the yield level (mean shift), 15% trim.
    Honest scope note: with short history only a single-break sup test is meaningful;
    full Bai-Perron multiple-break estimation is enabled once n_quarters is large."""
    y = np.asarray(y, dtype=float); n = len(y)
    trim = max(2, int(0.15*n))
    if n < 2*trim + 2:
        return dict(method="Quandt-Andrews supF (single break)", tested=False,
            reason="series too short for a break scan", breaks_detected=[])
    t = np.arange(n)
    Fs = []
    for k in range(trim, n-trim):
        d = (t>=k).astype(float)
        X = np.column_stack([np.ones(n), d])
        res = sm.OLS(y, X).fit()
        Fs.append((k, float(res.fvalue)))
    kbest, supF = max(Fs, key=lambda z: z[1])
    # Andrews (1993) asymptotic 5% critical value for 1 restriction, 15% trim ~ 8.85
    crit5 = 8.85
    detected = bool(supF > crit5)
    return dict(method="Quandt-Andrews supF (single break)", tested=True,
        supF=round(supF,3), crit_5pct=crit5, break_quarter_index=int(kbest),
        significant_at_5pct=detected, breaks_detected=([int(kbest)] if detected else []),
        note="Long-run yield is regime-dependent; a detected break means the pre-break sample "
             "should not be pooled into the long-run mean. Multiple-break Bai-Perron needs longer history.")

def fit_district(df, n_quarters, n_interp):
    logp = df["logp"].values; logr = df["logr"].values; yld = df["yield"].values
    coint = fit_cointegration(logp, logr)
    block = dict(
        n_quarters=int(n_quarters),
        date_range=[df.index[0], df.index[-1]],
        interpolated_quarters=int(n_interp),
        correlation=fit_correlation(logp, logr),
        cointegration=coint,
        vecm=fit_vecm(df, coint["cointegrated_at_5pct"]),
        ou_yield=fit_ou_yield(yld),
        structural_breaks=fit_structural_breaks(yld),
        regime="full_sample",
    )
    low_conf = n_quarters < MIN_RECOMMENDED_QUARTERS
    block["data_adequacy"] = dict(
        flag=("low_confidence" if low_conf else "ok"),
        reason=(f"{n_quarters} quarterly obs" + (" (single market regime)" if low_conf else "")),
        n_quarters=int(n_quarters), min_recommended_quarters=MIN_RECOMMENDED_QUARTERS)
    return block

def resolve_fit_level(sales, rent, districts, quarters_all):
    """Decide the econometric fit granularity. 'auto' picks the finest level where EVERY
    segment clears MIN_QUARTERS_FIT contiguous quarters; otherwise falls back to district.
    Today (~20q) this returns 'district'; a 20-year refresh auto-upgrades to 'district_band'
    with no code change (requirement: pipeline ready for longer history)."""
    if FIT_LEVEL in ("district", "district_band"):
        return FIT_LEVEL, f"forced by FIT_LEVEL={FIT_LEVEL!r}"
    # auto: subdivide into bands only if every band cell has enough REAL (non-interpolated)
    # quarters and isn't mostly interpolated. Counting real quarters (not the span length) is
    # what prevents a thin, hole-filled 20-quarter band series from being treated as fittable.
    ok_band = True; worst = None
    for d in districts:
        for b in BANDS:
            df, ni = district_quarterly(sales, rent, d, quarters_all, band=b)
            nq = 0 if df is None else len(df)
            real = nq - (ni or 0)
            frac = (ni/nq) if nq > 0 else 1.0
            if real < MIN_QUARTERS_FIT or frac > MAX_INTERP_FRAC_BAND:
                ok_band = False
                if worst is None or real < worst[2]: worst = (d, b, real, round(frac, 2))
    if ok_band:
        return "district_band", (f"auto: every band cell has >= {MIN_QUARTERS_FIT} real quarters "
                                 f"with <= {int(MAX_INTERP_FRAC_BAND*100)}% interpolation")
    return "district", (f"auto: fell back to district level - band cell {worst[0]}-{worst[1]} has "
                        f"{worst[2]} real quarters ({int(worst[3]*100)}% interpolated); needs "
                        f">= {MIN_QUARTERS_FIT} real and <= {int(MAX_INTERP_FRAC_BAND*100)}% interp")

def fit_segments(sales, rent, districts, quarters_all, fit_level):
    """Fit econometrics for each segment at the resolved level. Segment key is the district
    code (district level) or 'D09|B' (band level). MC is always district-level regardless."""
    segments = {}
    for d in districts:
        combos = [(d, None, d)] if fit_level == "district" else [(d, b, f"{d}|{b}") for b in BANDS]
        for dist, band, key in combos:
            df, ni = district_quarterly(sales, rent, dist, quarters_all, band=band)
            if df is None or len(df) < 3:
                continue
            blk = fit_district(df, len(df), ni)
            blk["segment"] = key; blk["district"] = dist; blk["band"] = band
            segments[key] = blk
    return segments

## 6. Finance helpers

Vectorized BSD, property tax, progressive income tax, and amortization - mirroring `app.js` / `build_data.py` so the Monte-Carlo terminal wealth is consistent with the live calculator.

In [7]:
_TAX_BRACKETS = [(0,0,0.0),(20000,0,0.02),(30000,200,0.035),(40000,550,0.07),(80000,3350,0.115),
    (120000,7950,0.15),(160000,13950,0.18),(200000,21150,0.19),(240000,28750,0.195),
    (280000,36550,0.20),(320000,44550,0.22),(500000,84150,0.23),(1000000,199150,0.24)]

def tax_on_income(x):
    x = np.asarray(x, dtype=float); out = np.zeros_like(x)
    for floor, base, rate in _TAX_BRACKETS:
        out = np.where(x >= floor, base + (x-floor)*rate, out)
    return np.maximum(out, 0.0)

def marginal_tax(base_income, slice_arr):
    slice_arr = np.maximum(np.asarray(slice_arr, dtype=float), 0.0)
    return tax_on_income(base_income + slice_arr) - tax_on_income(base_income)

def property_tax_annual(av):
    av = np.asarray(av, dtype=float)
    return np.where(av<=30000, av*0.12,
           np.where(av<=45000, 30000*0.12+(av-30000)*0.20,
           np.where(av<=60000, 30000*0.12+15000*0.20+(av-45000)*0.28,
                    30000*0.12+15000*0.20+15000*0.28+(av-60000)*0.36)))

def bsd(price):
    tiers = [(180000,0.01),(180000,0.02),(640000,0.03),(1000000,0.04),(1000000,0.05),(float('inf'),0.06)]
    rem, tot = price, 0.0
    for size, rate in tiers:
        amt = min(rem, size); tot += amt*rate; rem -= amt
        if rem <= 0: break
    return tot

def pmt(rm, n, pv):
    return pv*rm/(1-(1+rm)**-n) if rm>0 else pv/n

def amortization_yearly(loan, rate_pct, tenure_yrs, horizon_yrs):
    """Deterministic. Returns (interest_by_year[H], principal_by_year[H],
    payment_by_year[H], balance_at_horizon). Year index 0..H-1 = ownership years 1..H."""
    rm = rate_pct/100/12; N = int(round(tenure_yrs*12)); pay = pmt(rm, N, loan)
    bal = loan
    H = int(horizon_yrs)
    interest = np.zeros(H); principal = np.zeros(H); payment = np.zeros(H)
    for m in range(1, H*12+1):
        if bal <= 1e-9:
            break
        i = bal*rm; p = min(pay - i, bal); bal = max(bal - p, 0.0)
        yr = (m-1)//12
        interest[yr] += i; principal[yr] += p; payment[yr] += (i+p)
    return interest, principal, payment, bal

def ssd_rate(hold_years):
    if hold_years < 1: return 0.12
    if hold_years < 2: return 0.08
    if hold_years < 3: return 0.04
    return 0.0

## 7. Monte-Carlo engine

Correlated OU-yield + drifted-price simulation; full after-tax, after-exit-cost, inflation-adjusted (real) terminal wealth for buy vs ETF on every path.

In [8]:
def simulate_paths(rng, N, Q, y0, mu, lam, sigma, sigma_p, rho, P0, drift_pct):
    """Simulate correlated yield (OU) and price (GBM w/ user drift) paths.
    Returns yield[N,Q+1], price[N,Q+1] (t=0..Q). Reproducible given rng."""
    dt = DT_YEARS
    b = math.exp(-lam*dt)
    sig_step = sigma*math.sqrt((1-b*b)/(2*lam)) if lam>0 else sigma*math.sqrt(dt)
    # correlated shocks
    Zy = rng.standard_normal((N, Q))
    Zind = rng.standard_normal((N, Q))
    Zp = rho*Zy + math.sqrt(max(1-rho*rho,0.0))*Zind
    yld = np.empty((N, Q+1)); yld[:,0] = y0
    for t in range(1, Q+1):
        yld[:,t] = mu + b*(yld[:,t-1]-mu) + sig_step*Zy[:,t-1]
    yld = np.clip(yld, 1e-4, None)
    mu_p_q = math.log(1+drift_pct/100)/4.0
    dlogp = mu_p_q + sigma_p*Zp
    logP = np.concatenate([np.full((N,1), math.log(P0)), math.log(P0)+np.cumsum(dlogp, axis=1)], axis=1)
    price = np.exp(logP)
    return yld, price

def yearly_rent(yld, price, horizon):
    """Annual rent (=yield*price) averaged over the 4 quarters of each ownership year -> [N,horizon].
    Vectorized: ownership year k (0-based) uses quarter columns 4k+1..4k+4."""
    rent_annual = yld*price                                  # [N,Q+1], annual rent at each quarter
    ra = rent_annual[:, 1:4*horizon+1]                       # [N,4H], drop t0
    return ra.reshape(ra.shape[0], horizon, 4).mean(axis=2)  # [N,H]

def aftertax_cashflows(rent_year, interest_y, payment_y):
    """Per-year after-tax rental cash flow [N,H]. rent_year[N,H] full annual market rent;
    interest_y[H]/payment_y[H] deterministic. Opex grows at EXPENSE_GROWTH_PCT; rent grows
    via the simulated yield*price path (not frozen). Mirrors app.js cashFlowForYear."""
    N, H = rent_year.shape
    g = EXPENSE_GROWTH_PCT/100
    eff = rent_year*(12-VAC_MO)/12.0
    yrs = np.arange(H)
    grow = (1+g)**yrs                                  # [H]
    mcst = (MCST_MO*12)*grow; ins = INS_YR*grow        # [H] fixed opex, inflating
    agent = eff*AGENT_PCT; repair = eff*REPAIR_PCT      # [N,H] % of rent
    ptax = property_tax_annual(rent_year)               # [N,H] on full annual rent
    pretax = eff - payment_y[None,:] - mcst[None,:] - ins[None,:] - agent - repair - ptax
    taxable = eff - interest_y[None,:] - mcst[None,:] - ins[None,:] - agent - repair - ptax
    inctax = marginal_tax(BASE_INCOME, taxable)
    return pretax - inctax                              # [N,H] after-tax annual cash flow

def terminal_wealth(price, rent_year_full, ltv_pct, horizon, etf_pct, P0, amort_cache):
    """Full after-tax, after-exit-cost, REAL terminal wealth for buy vs ETF. Vectorized [N].
    Mirrors the app.js opportunity-cost methodology (net equity + reinvested surplus vs
    outlay grown at ETF + redirected shortfalls), plus exit costs, opex/rent growth, real adj."""
    N = price.shape[0]
    loan = P0*ltv_pct/100.0
    key = (round(loan,2), horizon)
    if key not in amort_cache:
        amort_cache[key] = amortization_yearly(loan, MORTGAGE_RATE_PCT, LOAN_TENURE_YRS, horizon)
    interest_y, principal_y, payment_y, bal_T = amort_cache[key]
    cf = aftertax_cashflows(rent_year_full[:,:horizon], interest_y, payment_y)   # [N,H]
    etf = etf_pct/100.0
    factors = (1+etf)**(horizon - (np.arange(horizon)+1))                        # [H] compound each yr-end to T
    surplus = np.where(cf>0, cf, 0.0) @ factors
    shortfall = np.where(cf<0, -cf, 0.0) @ factors
    P_T = price[:, 4*horizon]
    sell_comm = SELL_AGENT_PCT*P_T
    ssd = ssd_rate(horizon)*P_T
    buy_nom = (P_T - bal_T - sell_comm - ssd) + surplus
    downpayment = P0*(1-ltv_pct/100.0)
    outlay = downpayment + bsd(P0) + P0*ABSD_PCT + LEGAL_MISC
    etf_nom = outlay*(1+etf)**horizon + shortfall
    infl = (1+INFLATION_PCT/100.0)**horizon
    return buy_nom/infl, etf_nom/infl, outlay

def pctile_summary(arr):
    # integer dollars rounded to the nearest 100 - keeps mc_grid.json small; the MC sampling
    # noise on these percentiles is far larger than $100, so no meaningful precision is lost.
    p = np.percentile(arr, [10,25,50,75,90])
    return dict(p10=int(round(float(p[0]),-2)), p25=int(round(float(p[1]),-2)),
                median=int(round(float(p[2]),-2)), p75=int(round(float(p[3]),-2)), p90=int(round(float(p[4]),-2)))

## 8. Drivers

In [9]:
def district_mc_inputs(sales, rent, district, quarters_all):
    """Reference price, price vol, price-yield shock corr, OU params, yield buckets.
    The MC is ALWAYS district-level regardless of the model_params fit level, so the OU is
    (re)calibrated here from the district-pooled series rather than read from a segment block -
    a per-band OU would be the wrong input for a district-level simulation."""
    d_sales = sales[sales["district"]==district]
    P0 = float(np.median(d_sales["price"]))
    df, _ = district_quarterly(sales, rent, district, quarters_all)
    dlogp = np.diff(df["logp"].values)
    sigma_p = float(np.std(dlogp, ddof=1)) if len(dlogp)>1 else 0.02
    dyield = np.diff(df["yield"].values)
    m = min(len(dlogp), len(dyield))
    rho = float(np.corrcoef(dlogp[:m], dyield[:m])[0,1]) if m>2 and np.std(dyield[:m])>0 else 0.0
    ou = fit_ou_yield(df["yield"].values)   # district-pooled OU, always
    yq = df["yield"].values
    edges = np.quantile(yq, [0, 1/3, 2/3, 1.0])
    buckets = {
        "low":  {"y0": float(np.mean(yq[yq<=edges[1]])) if (yq<=edges[1]).any() else float(edges[0]), "edges":[round(float(edges[0]),4), round(float(edges[1]),4)]},
        "mid":  {"y0": float(np.mean(yq[(yq>edges[1])&(yq<=edges[2])])) if ((yq>edges[1])&(yq<=edges[2])).any() else float(edges[1]), "edges":[round(float(edges[1]),4), round(float(edges[2]),4)]},
        "high": {"y0": float(np.mean(yq[yq>edges[2]])) if (yq>edges[2]).any() else float(edges[3]), "edges":[round(float(edges[2]),4), round(float(edges[3]),4)]},
    }
    return dict(P0=round(P0,0), sigma_p=round(sigma_p,5), rho_price_yield=round(rho,4),
                mu=ou["long_run_yield"], lam=ou["lambda_"], sigma=ou["sigma"], buckets=buckets)

# --- optimized path pieces: generate the STOCHASTIC part once, then apply any drift cheaply ---
def _ou_path(Zy, y0, mu, lam, sigma):
    """OU yield path from pre-drawn shocks Zy[N,Q]. Same exact discretization as simulate_paths."""
    N, Q = Zy.shape
    dt = DT_YEARS; b = math.exp(-lam*dt)
    sig_step = sigma*math.sqrt((1-b*b)/(2*lam)) if lam > 0 else sigma*math.sqrt(dt)
    yld = np.empty((N, Q+1)); yld[:,0] = y0
    for t in range(1, Q+1):
        yld[:,t] = mu + b*(yld[:,t-1]-mu) + sig_step*Zy[:,t-1]
    return np.clip(yld, 1e-4, None)

def _price_stoch_cum(Zp, sigma_p):
    """Cumulative zero-drift log price deviation [N,Q+1] (col 0 = 0)."""
    N, Q = Zp.shape
    return np.concatenate([np.zeros((N,1)), np.cumsum(sigma_p*Zp, axis=1)], axis=1)

def _price_from(P0, stoch_cum, drift_pct):
    """Price path for a given drift = P0 * exp(drift_cum + stochastic_cum). Cheap (one exp)."""
    Qp1 = stoch_cum.shape[1]
    mu_p_q = math.log(1+drift_pct/100)/4.0
    return P0*np.exp(mu_p_q*np.arange(Qp1)[None,:] + stoch_cum)

def run_district(mc_in, seed):
    """Full per-district MC. Draws shocks ONCE (common random numbers), reuses them across
    every grid cell and the breakeven bisection, so price for any drift is a single exp().
    Returns (cells, breakeven)."""
    Qmax = max(AX_HORIZON)*4; N = MC_N_PATHS; P0 = mc_in["P0"]
    rng = np.random.default_rng(seed)
    Zy = rng.standard_normal((N, Qmax))            # drawn first (matches simulate_paths order)
    Zind = rng.standard_normal((N, Qmax))
    rho = mc_in["rho_price_yield"]
    Zp = rho*Zy + math.sqrt(max(1-rho*rho, 0.0))*Zind
    stoch_cum = _price_stoch_cum(Zp, mc_in["sigma_p"])
    yld_by_bucket = {b: _ou_path(Zy, mc_in["buckets"][b]["y0"], mc_in["mu"], mc_in["lam"], mc_in["sigma"])
                     for b in AX_YIELD_BUCKET}
    price_by_drift = {dr: _price_from(P0, stoch_cum, dr) for dr in AX_DRIFT}
    amort_cache = {}

    cells = []
    for bucket in AX_YIELD_BUCKET:
        yld = yld_by_bucket[bucket]
        for drift in AX_DRIFT:
            price = price_by_drift[drift]
            rent_year_full = yearly_rent(yld, price, max(AX_HORIZON))
            for horizon in AX_HORIZON:
                for ltv in AX_LTV:
                    for etf in AX_ETF:
                        buy, etfw, outlay = terminal_wealth(price, rent_year_full, ltv, horizon, etf, P0, amort_cache)
                        cells.append(dict(horizon_years=horizon, ltv_pct=ltv, entry_yield_bucket=bucket,
                            appreciation_drift_pct=drift, etf_return_pct=etf,
                            ref_price=int(P0), outlay=int(round(outlay,-2)),
                            buy_terminal_wealth_real=pctile_summary(buy),
                            etf_terminal_wealth_real=pctile_summary(etfw),
                            prob_buy_beats_etf=round(float((buy>etfw).mean()),4)))

    def median_diff(yld, drift, horizon, ltv, etf):
        Qh = 4*horizon                                       # only simulate to the horizon width
        price = _price_from(P0, stoch_cum[:, :Qh+1], drift)
        ry = yearly_rent(yld[:, :Qh+1], price, horizon)
        buy, etfw, _ = terminal_wealth(price, ry, ltv, horizon, etf, P0, amort_cache)
        return np.median(buy) - np.median(etfw)

    breakeven = []
    for bucket in AX_YIELD_BUCKET:
        yld = yld_by_bucket[bucket]
        for horizon in AX_HORIZON:
            for ltv in AX_LTV:
                for etf in AX_ETF:
                    lo, hi = -10.0, 25.0
                    flo = median_diff(yld, lo, horizon, ltv, etf)
                    fhi = median_diff(yld, hi, horizon, ltv, etf)
                    if flo > 0:   be = lo     # buy already wins even at -10% drift
                    elif fhi < 0: be = None    # buy never catches up within range
                    else:
                        for _ in range(18):   # 35%-range / 2^18 ~ 1e-4%, well under MC median noise
                            mid = (lo+hi)/2; fm = median_diff(yld, mid, horizon, ltv, etf)
                            if fm > 0: hi = mid
                            else: lo = mid
                        be = (lo+hi)/2
                    breakeven.append(dict(horizon_years=horizon, ltv_pct=ltv, entry_yield_bucket=bucket,
                        etf_return_pct=etf, breakeven_appreciation_cagr_pct=(round(be,2) if be is not None else None)))
    return cells, breakeven

## Self-Checks

## 9. Run the pipeline and write artifacts

In [10]:
import warnings; warnings.filterwarnings("ignore")
os.makedirs(OUT_DIR, exist_ok=True)

sales, rent = load_data()
sales, rent = add_quarter_band(sales, rent)
print(f"Loaded {len(sales):,} sales rows, {len(rent):,} rent rows.")

series_obj, quarters_all, districts = build_series(sales, rent)
with open(os.path.join(OUT_DIR, "series.json"), "w") as f:
    json.dump(series_obj, f)
print(f"series.json: {len(series_obj['series'])} district x band series over "
      f"{len(quarters_all)} quarters ({quarters_all[0]}..{quarters_all[-1]}).")

Loaded 17,164 sales rows, 74,697 rent rows.
series.json: 21 district x band series over 22 quarters (2021-Q2..2026-Q3).


In [11]:
# --- model_params.json: econometric fits at the resolved level ---
fit_level, fit_reason = resolve_fit_level(sales, rent, districts, quarters_all)
print("FIT_LEVEL resolved to:", fit_level, "->", fit_reason)
segments = fit_segments(sales, rent, districts, quarters_all, fit_level)

any_ok = any(b["data_adequacy"]["flag"] == "ok" for b in segments.values())
model_params = dict(
    fit_level=fit_level, fit_level_reason=fit_reason,
    segments=segments,
    global_data_adequacy=dict(
        flag=("ok" if any_ok else "low_confidence"),
        reason=(f"{min(b['n_quarters'] for b in segments.values())}-"
                f"{max(b['n_quarters'] for b in segments.values())} quarterly obs per segment; "
                + ("some segments meet the minimum" if any_ok else
                   f"all below {MIN_RECOMMENDED_QUARTERS} - cointegration/VECM/OU are provisional, not robust"))),
)
with open(os.path.join(OUT_DIR, "model_params.json"), "w") as f:
    json.dump(model_params, f, indent=1)
for k, b in segments.items():
    c = b["cointegration"]["cointegrated_at_5pct"]; ou = b["ou_yield"]
    print(f"  {k}: n={b['n_quarters']} coint={c} OU mu={ou['long_run_yield']} "
          f"lambda={ou['lambda_']} flag={b['data_adequacy']['flag']}")

FIT_LEVEL resolved to: district -> auto: fell back to district level - band cell D04-B has 14 real quarters (30% interpolated); needs >= 40 real and <= 15% interp
  D01: n=20 coint=False OU mu=0.03731 lambda=1.3947 flag=low_confidence
  D02: n=20 coint=True OU mu=0.0365 lambda=9.2212 flag=low_confidence
  D03: n=20 coint=True OU mu=0.03606 lambda=1.3918 flag=low_confidence
  D04: n=20 coint=False OU mu=0.03574 lambda=0.9808 flag=low_confidence
  D09: n=20 coint=False OU mu=0.03103 lambda=2.6771 flag=low_confidence
  D10: n=20 coint=True OU mu=0.02875 lambda=1.6177 flag=low_confidence
  D15: n=20 coint=True OU mu=0.03042 lambda=8.0509 flag=low_confidence


In [12]:
# --- mc_grid.json: precomputed Monte-Carlo grid + break-even (district-level MC) ---
import time
all_cells, all_be, bucket_edges = [], [], {}
t0 = time.time()
for d in districts:
    mc_in = district_mc_inputs(sales, rent, d, quarters_all)   # always district-level OU
    bucket_edges[d] = {b: mc_in["buckets"][b]["edges"] for b in AX_YIELD_BUCKET}
    cells, be = run_district(mc_in, MC_SEED)
    for c in cells: c["district"] = d
    for b in be:    b["district"] = d
    all_cells += cells; all_be += be
    print(f"  {d}: {len(cells)} cells, {len(be)} break-even  ({time.time()-t0:.0f}s elapsed)")

mc_grid = dict(
    axes=dict(district=districts, horizon_years=AX_HORIZON, ltv_pct=AX_LTV,
              entry_yield_bucket=AX_YIELD_BUCKET, appreciation_drift_pct=AX_DRIFT, etf_return_pct=AX_ETF),
    entry_yield_bucket_edges=bucket_edges,
    band_price_scaling=dict(note="MC is district-level; band affects the DOLLAR level (stamp-duty tiers, "
        "loan size) via deterministic client-side arithmetic, not the simulated return path."),
    reference_buyer=dict(absd_pct=ABSD_PCT*100, note="SC 1st property; dollar outputs are provisional and scale with the user's own price/ABSD."),
    cells=all_cells, breakeven=all_be)
with open(os.path.join(OUT_DIR, "mc_grid.json"), "w") as f:
    json.dump(mc_grid, f)
print(f"mc_grid.json: {len(all_cells)} cells + {len(all_be)} break-even rows in {time.time()-t0:.0f}s.")

  D01: 4050 cells, 810 break-even  (86s elapsed)


  D02: 4050 cells, 810 break-even  (171s elapsed)


  D03: 4050 cells, 810 break-even  (257s elapsed)


  D04: 4050 cells, 810 break-even  (341s elapsed)


  D09: 4050 cells, 810 break-even  (426s elapsed)


  D10: 4050 cells, 810 break-even  (510s elapsed)


  D15: 4050 cells, 810 break-even  (594s elapsed)


mc_grid.json: 28350 cells + 5670 break-even rows in 595s.


In [13]:
# --- model_meta.json: provenance ---
import datetime
meta = dict(
    generated=datetime.datetime.now().isoformat(timespec="seconds"),
    notebook="sg_price_rent_model.ipynb",
    source=dict(kind=SOURCE_KIND,
                sales_file=os.path.basename(HISTORICAL_SALES_SOURCE),
                rent_file=os.path.basename(HISTORICAL_RENT_SOURCE),
                date_range=[quarters_all[0], quarters_all[-1]], n_quarters=len(quarters_all),
                districts=districts),
    model_specs=dict(
        fit_level=fit_level,
        cointegration=["engle_granger (fixed lag)", "johansen (k_ar_diff=1)"],
        vecm="rank 1, k_ar_diff=1, deterministic='ci'; error-correction + rent->price Granger",
        ou_calibration="AR(1)-on-yield OLS, exact discretization (dt=0.25yr)",
        structural_breaks="Quandt-Andrews supF single-break, 15% trim"),
    monte_carlo=dict(n_paths=MC_N_PATHS, seed=MC_SEED, dt_years=DT_YEARS,
        grid_axes=dict(horizon_years=AX_HORIZON, ltv_pct=AX_LTV, entry_yield_bucket=AX_YIELD_BUCKET,
                       appreciation_drift_pct=AX_DRIFT, etf_return_pct=AX_ETF),
        finance=dict(mortgage_rate_pct=MORTGAGE_RATE_PCT, loan_tenure_yrs=LOAN_TENURE_YRS,
            mcst_month=MCST_MO, insurance_yr=INS_YR, agent_pct=AGENT_PCT*100, repair_pct=REPAIR_PCT*100,
            vacancy_months=VAC_MO, base_income=BASE_INCOME, sell_agent_pct=SELL_AGENT_PCT*100,
            inflation_pct=INFLATION_PCT, expense_growth_pct=EXPENSE_GROWTH_PCT)),
    data_adequacy_global=model_params["global_data_adequacy"],
)
with open(os.path.join(OUT_DIR, "model_meta.json"), "w") as f:
    json.dump(meta, f, indent=2)
print("model_meta.json written.")
print(json.dumps(meta["data_adequacy_global"], indent=2))

model_meta.json written.
{
  "flag": "low_confidence",
  "reason": "20-20 quarterly obs per segment; all below 40 - cointegration/VECM/OU are provisional, not robust"
}


## 10. Done

All four artifacts are in `data/`. Each is independently shippable. To diff fitted parameters
across data refreshes (catch instability), compare successive `model_params.json` / `model_meta.json`.